# 02. Construccion de Agronet anual limpio

Este notebook reconstruye la capa anual de Agronet a nivel `departamento-anio` a partir del archivo municipal crudo. Su objetivo es dejar una base productiva anual consistente, trazable y lista para alimentar la construccion del dataset principal de modelado.

## Objetivos
- cargar y auditar la fuente cruda `agronet_cafe.xlsx`
- normalizar nombres de columnas y formatos numericos
- agregar la informacion a nivel `departamento-anio`
- revisar el caso atipico de Cundinamarca 2008
- aplicar una correccion explicita y trazable para ese caso
- recalcular variables derivadas despues de la correccion
- exportar una base anual limpia a `BASE_DE_DATOS/INTERMEDIAS`

## Resultado esperado
Al finalizar este notebook debemos tener un archivo intermedio con:

- una fila por `departamento-anio`
- medidas originales y corregidas cuando aplique
- una marca explicita de la correccion de 2008
- rendimiento anual recalculado
- medias historicas departamentales consistentes
- una base que sirva como capa productiva observada del proyecto


## Criterio metodologico de este notebook

Este paso no construye todavia el dataset final de modelado. Aqui solo dejamos bien resuelta la fuente productiva anual.

La correccion de Cundinamarca 2008 se mantiene como una decision explicita del proyecto porque ya fue detectada en los notebooks anteriores como un outlier biologicamente improbable. En vez de ocultarla, este notebook:

- conserva los valores originales
- crea columnas corregidas
- marca la fila afectada
- recalcula todo lo derivado despues de aplicar la correccion

De esta forma queda trazabilidad completa entre dato original y dato ajustado.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)
pd.set_option('display.max_colwidth', 140)


def find_project_root(start: Path) -> Path:
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / 'BASE_DE_DATOS').exists():
            return candidate
    raise FileNotFoundError('No se encontro una carpeta BASE_DE_DATOS en la ruta actual ni en sus padres.')


CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = find_project_root(CURRENT_DIR)
BASE_DATOS = PROJECT_ROOT / 'BASE_DE_DATOS'
INPUT_PATH = BASE_DATOS / 'ENTRADA' / 'agronet_cafe.xlsx'
OUTPUT_DIR = BASE_DATOS / 'INTERMEDIAS'
OUTPUT_PATH = OUTPUT_DIR / 'agronet_departamento_anual.csv'

print(f'Ruta actual: {CURRENT_DIR}')
print(f'Raiz detectada del proyecto: {PROJECT_ROOT}')
print(f'Archivo de entrada: {INPUT_PATH}')
print(f'Archivo de salida: {OUTPUT_PATH}')


Ruta actual: C:\Users\crist\Documents\MAESTRIA\PROYECTO_GRADO\MODELOS
Raiz detectada del proyecto: C:\Users\crist\Documents\MAESTRIA\PROYECTO_GRADO
Archivo de entrada: C:\Users\crist\Documents\MAESTRIA\PROYECTO_GRADO\BASE_DE_DATOS\ENTRADA\agronet_cafe.xlsx
Archivo de salida: C:\Users\crist\Documents\MAESTRIA\PROYECTO_GRADO\BASE_DE_DATOS\INTERMEDIAS\agronet_departamento_anual.csv


In [2]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f'No existe el archivo esperado: {INPUT_PATH}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Rutas validadas correctamente.')


Rutas validadas correctamente.


## Carga del archivo crudo de Agronet

Primero leemos el archivo tal como viene y verificamos hoja, columnas y una muestra inicial.


In [3]:
xls = pd.ExcelFile(INPUT_PATH)
sheet_name = xls.sheet_names[0]
agronet_raw = pd.read_excel(INPUT_PATH, sheet_name=sheet_name)

print('Hojas disponibles:', xls.sheet_names)
print('Hoja usada:', sheet_name)
print('Shape inicial:', agronet_raw.shape)
print('Columnas originales:', agronet_raw.columns.tolist())
display(agronet_raw.head(5))


Hojas disponibles: ['detalle_agricola_municipal_muni']
Hoja usada: detalle_agricola_municipal_muni
Shape inicial: (1483, 9)
Columnas originales: ['Año', 'Departamento', 'Municipio', 'Producto', 'Área Cosechada', 'Área Sembrada', 'Producción', 'Rendimiento', 'Ciclo']


,Año,Departamento,Municipio,Producto,Área Cosechada,Área Sembrada,Producción,Rendimiento,Ciclo
0,2019,RISARALDA,APIA,CAFE,3266.00,4338.00,4335.00,1.327312,PERMANENTE
1,2014,RISARALDA,APIA,CAFE,3922.46,5061.94,3399.13,0.866581,PERMANENTE
2,2007,RISARALDA,APIA,CAFE,4451.00,4851.00,7410.60,1.664929,PERMANENTE
3,2011,RISARALDA,APIA,CAFE,4088.50,4904.78,5108.48,1.249475,PERMANENTE
4,2015,RISARALDA,APIA,CAFE,4031.91,5083.71,4743.99,1.176611,PERMANENTE


## Normalizacion de columnas y textos

Aqui homologamos nombres para trabajar con una convención simple y reproducible en el resto del proyecto.


In [4]:
rename_map = {
    'Año': 'anio',
    'Departamento': 'departamento',
    'Municipio': 'municipio',
    'Producto': 'producto',
    'Área Cosechada': 'area_cosechada_ha',
    'Área Sembrada': 'area_sembrada_ha',
    'Producción': 'produccion_t',
    'Rendimiento': 'rendimiento_t_ha_reportado',
    'Ciclo': 'ciclo',
}

agronet = agronet_raw.rename(columns=rename_map).copy()

for col in ['departamento', 'municipio', 'producto', 'ciclo']:
    agronet[col] = agronet[col].astype(str).str.strip()

agronet['departamento'] = agronet['departamento'].str.title()
agronet['municipio'] = agronet['municipio'].str.title()
agronet['producto'] = agronet['producto'].str.upper()
agronet['ciclo'] = agronet['ciclo'].str.upper()

numeric_cols = ['anio', 'area_cosechada_ha', 'area_sembrada_ha', 'produccion_t', 'rendimiento_t_ha_reportado']
for col in numeric_cols:
    agronet[col] = pd.to_numeric(agronet[col], errors='coerce')

print('Shape despues de normalizar:', agronet.shape)
display(agronet.head(5))


Shape despues de normalizar: (1483, 9)


,anio,departamento,municipio,producto,area_cosechada_ha,area_sembrada_ha,produccion_t,rendimiento_t_ha_reportado,ciclo
0,2019,Risaralda,Apia,CAFE,3266.00,4338.00,4335.00,1.327312,PERMANENTE
1,2014,Risaralda,Apia,CAFE,3922.46,5061.94,3399.13,0.866581,PERMANENTE
2,2007,Risaralda,Apia,CAFE,4451.00,4851.00,7410.60,1.664929,PERMANENTE
3,2011,Risaralda,Apia,CAFE,4088.50,4904.78,5108.48,1.249475,PERMANENTE
4,2015,Risaralda,Apia,CAFE,4031.91,5083.71,4743.99,1.176611,PERMANENTE


## Chequeos basicos del archivo crudo

Antes de agregar, revisamos que el archivo tenga exactamente el alcance que esperamos para el proyecto.


In [5]:
chequeos_base = {
    'anios_min_max': (int(agronet['anio'].min()), int(agronet['anio'].max())),
    'departamentos': sorted(agronet['departamento'].dropna().unique().tolist()),
    'productos': sorted(agronet['producto'].dropna().unique().tolist()),
    'ciclos': sorted(agronet['ciclo'].dropna().unique().tolist()),
    'nulos_top': agronet.isnull().sum().sort_values(ascending=False).head(10).to_dict(),
}

print(json.dumps(chequeos_base, indent=2, ensure_ascii=False))


{
  "anios_min_max": [
    2007,
    2024
  ],
  "departamentos": [
    "Cundinamarca",
    "Risaralda"
  ],
  "productos": [
    "CAFE"
  ],
  "ciclos": [
    "PERMANENTE"
  ],
  "nulos_top": {
    "anio": 0,
    "departamento": 0,
    "municipio": 0,
    "producto": 0,
    "area_cosechada_ha": 0,
    "area_sembrada_ha": 0,
    "produccion_t": 0,
    "rendimiento_t_ha_reportado": 0,
    "ciclo": 0
  }
}


In [6]:
resumen_departamento_anio_muni = (
    agronet.groupby(['departamento', 'anio'])['municipio']
    .nunique()
    .reset_index(name='n_municipios')
)

display(resumen_departamento_anio_muni.head(10))


,departamento,anio,n_municipios
0,Cundinamarca,2007,65
1,Cundinamarca,2008,66
2,Cundinamarca,2009,69
3,Cundinamarca,2010,69
4,Cundinamarca,2011,69
5,Cundinamarca,2012,68
6,Cundinamarca,2013,70
7,Cundinamarca,2014,69
8,Cundinamarca,2015,69
9,Cundinamarca,2016,69


## Agregacion a nivel `departamento-anio`

Agronet viene a nivel municipal. Aqui lo llevamos al nivel anual departamental que necesitamos como capa productiva observada del proyecto.


In [7]:
agronet_anual = (
    agronet.groupby(['anio', 'departamento'], as_index=False)
    .agg(
        n_municipios=('municipio', 'nunique'),
        area_cosechada_ha=('area_cosechada_ha', 'sum'),
        area_sembrada_ha=('area_sembrada_ha', 'sum'),
        produccion_t=('produccion_t', 'sum'),
        rendimiento_medio_municipal_reportado=('rendimiento_t_ha_reportado', 'mean'),
    )
)

agronet_anual['rendimiento_t_ha'] = agronet_anual['produccion_t'] / agronet_anual['area_cosechada_ha']

print('Shape anual departamental:', agronet_anual.shape)
display(agronet_anual.head(8))


Shape anual departamental: (36, 8)


,anio,departamento,n_municipios,area_cosechada_ha,area_sembrada_ha,produccion_t,rendimiento_medio_municipal_reportado,rendimiento_t_ha
0,2007,Cundinamarca,65,43017.30,48195.69,33729.143730,0.629565,0.784083
1,2007,Risaralda,14,47689.25,55714.64,72842.500000,1.444499,1.527441
2,2008,Cundinamarca,66,43633.35,48989.09,78254.745626,1.908602,1.793462
3,2008,Risaralda,14,47227.00,55074.75,60079.000000,1.142505,1.272132
4,2009,Cundinamarca,69,43475.84,48581.30,37118.057049,0.751595,0.853763
5,2009,Risaralda,14,45428.00,54252.96,53648.000000,1.099386,1.180946
6,2010,Cundinamarca,69,44264.16,49357.78,37214.800000,0.783393,0.840743
7,2010,Risaralda,14,47308.00,52897.00,72091.000000,1.447109,1.523865


## Revision de consistencia del rendimiento anual

Comparamos el rendimiento agregado calculado con el promedio simple del rendimiento municipal reportado. Para este proyecto, la medida que usaremos es la calculada como `produccion / area_cosechada`, porque representa correctamente el agregado departamental.


In [8]:
agronet_anual['dif_rendimiento_calculado_vs_reportado'] = (
    agronet_anual['rendimiento_t_ha'] - agronet_anual['rendimiento_medio_municipal_reportado']
)

display(
    agronet_anual[
        ['anio', 'departamento', 'n_municipios', 'produccion_t', 'area_cosechada_ha', 'rendimiento_t_ha', 'rendimiento_medio_municipal_reportado', 'dif_rendimiento_calculado_vs_reportado']
    ].head(10)
)


,anio,departamento,n_municipios,produccion_t,area_cosechada_ha,rendimiento_t_ha,rendimiento_medio_municipal_reportado,dif_rendimiento_calculado_vs_reportado
0,2007,Cundinamarca,65,33729.143730,43017.30,0.784083,0.629565,0.154518
1,2007,Risaralda,14,72842.500000,47689.25,1.527441,1.444499,0.082941
2,2008,Cundinamarca,66,78254.745626,43633.35,1.793462,1.908602,-0.115141
3,2008,Risaralda,14,60079.000000,47227.00,1.272132,1.142505,0.129628
4,2009,Cundinamarca,69,37118.057049,43475.84,0.853763,0.751595,0.102168
5,2009,Risaralda,14,53648.000000,45428.00,1.180946,1.099386,0.081560
6,2010,Cundinamarca,69,37214.800000,44264.16,0.840743,0.783393,0.057350
7,2010,Risaralda,14,72091.000000,47308.00,1.523865,1.447109,0.076756
8,2011,Cundinamarca,69,32780.348700,37478.87,0.874635,0.854516,0.020120
9,2011,Risaralda,14,49042.310000,44733.64,1.096318,1.047150,0.049168


## Deteccion del caso atipico de Cundinamarca 2008

El objetivo aqui no es aplicar una regla automatica ciega, sino revisar el caso ya identificado en el proyecto y dejar evidencia de por que se corrige.


In [9]:
cundi = agronet_anual[agronet_anual['departamento'] == 'Cundinamarca'].copy()
cundi['zscore_rendimiento'] = (
    (cundi['rendimiento_t_ha'] - cundi['rendimiento_t_ha'].mean()) / cundi['rendimiento_t_ha'].std(ddof=0)
)

display(cundi[['anio', 'departamento', 'area_cosechada_ha', 'produccion_t', 'rendimiento_t_ha', 'zscore_rendimiento']].sort_values('anio'))


,anio,departamento,area_cosechada_ha,produccion_t,rendimiento_t_ha,zscore_rendimiento
0,2007,Cundinamarca,43017.30000,33729.143730,0.784083,-0.803832
2,2008,Cundinamarca,43633.35000,78254.745626,1.793462,3.353299
4,2009,Cundinamarca,43475.84000,37118.057049,0.853763,-0.516856
6,2010,Cundinamarca,44264.16000,37214.800000,0.840743,-0.570477
8,2011,Cundinamarca,37478.87000,32780.348700,0.874635,-0.430892
10,2012,Cundinamarca,37175.06000,30786.406900,0.828147,-0.622356
12,2013,Cundinamarca,36189.18000,24993.744755,0.690641,-1.188673
14,2014,Cundinamarca,33623.54000,25118.557250,0.747053,-0.956343
16,2015,Cundinamarca,34101.49000,31165.162260,0.913894,-0.269204
18,2016,Cundinamarca,33214.16263,31413.339460,0.945781,-0.137877


In [10]:
cundi_2008 = cundi[cundi['anio'] == 2008].iloc[0]
cundi_sin_2008 = cundi[cundi['anio'] != 2008]

diagnostico_2008 = {
    'rendimiento_2008': float(cundi_2008['rendimiento_t_ha']),
    'max_historico_sin_2008': float(cundi_sin_2008['rendimiento_t_ha'].max()),
    'promedio_historico_sin_2008': float(cundi_sin_2008['rendimiento_t_ha'].mean()),
    'razon_2008_vs_max_sin_2008': float(cundi_2008['rendimiento_t_ha'] / cundi_sin_2008['rendimiento_t_ha'].max()),
    'zscore_2008': float(cundi_2008['zscore_rendimiento']),
}

print(json.dumps(diagnostico_2008, indent=2, ensure_ascii=False))


{
  "rendimiento_2008": 1.7934617815446212,
  "max_historico_sin_2008": 1.1780109204368174,
  "promedio_historico_sin_2008": 0.9313646610350608,
  "razon_2008_vs_max_sin_2008": 1.5224491984162498,
  "zscore_2008": 3.3532992390618146
}


## Regla de correccion adoptada para Cundinamarca 2008

Se usa una imputacion local y trazable basada en los años vecinos del mismo departamento:

- rendimiento corregido 2008 = promedio de rendimiento 2007 y 2009
- produccion corregida 2008 = rendimiento corregido x area cosechada 2008

Este criterio conserva la escala local del departamento y evita introducir una regla externa dificil de defender.


In [11]:
rend_2007 = cundi.loc[cundi['anio'] == 2007, 'rendimiento_t_ha'].iloc[0]
rend_2009 = cundi.loc[cundi['anio'] == 2009, 'rendimiento_t_ha'].iloc[0]
rendimiento_corregido_2008 = (rend_2007 + rend_2009) / 2

area_2008 = cundi.loc[cundi['anio'] == 2008, 'area_cosechada_ha'].iloc[0]
produccion_corregida_2008 = rendimiento_corregido_2008 * area_2008

correccion_2008 = {
    'rend_2007': float(rend_2007),
    'rend_2009': float(rend_2009),
    'rendimiento_corregido_2008': float(rendimiento_corregido_2008),
    'area_cosechada_2008': float(area_2008),
    'produccion_corregida_2008': float(produccion_corregida_2008),
}

print(json.dumps(correccion_2008, indent=2, ensure_ascii=False))


{
  "rend_2007": 0.7840832346583816,
  "rend_2009": 0.8537628496424681,
  "rendimiento_corregido_2008": 0.8189230421504248,
  "area_cosechada_2008": 43633.35,
  "produccion_corregida_2008": 35732.35572121423
}


## Aplicacion de la correccion y recálculo integral

Aqui preservamos los valores originales y creamos la version corregida que se utilizara en las siguientes etapas del pipeline.


In [12]:
agronet_anual_corr = agronet_anual.copy()

agronet_anual_corr = agronet_anual_corr.rename(columns={
    'produccion_t': 'produccion_t_original',
    'rendimiento_t_ha': 'rendimiento_t_ha_original',
})

agronet_anual_corr['correccion_aplicada'] = 0
agronet_anual_corr['motivo_correccion'] = pd.NA

agronet_anual_corr['produccion_t'] = agronet_anual_corr['produccion_t_original']
agronet_anual_corr['rendimiento_t_ha'] = agronet_anual_corr['rendimiento_t_ha_original']

mask_2008_cundi = (
    (agronet_anual_corr['departamento'] == 'Cundinamarca') &
    (agronet_anual_corr['anio'] == 2008)
)

agronet_anual_corr.loc[mask_2008_cundi, 'produccion_t'] = produccion_corregida_2008
agronet_anual_corr.loc[mask_2008_cundi, 'rendimiento_t_ha'] = rendimiento_corregido_2008
agronet_anual_corr.loc[mask_2008_cundi, 'correccion_aplicada'] = 1
agronet_anual_corr.loc[mask_2008_cundi, 'motivo_correccion'] = 'Outlier de rendimiento de Cundinamarca 2008 corregido con promedio 2007-2009'

agronet_anual_corr['delta_produccion_t'] = agronet_anual_corr['produccion_t'] - agronet_anual_corr['produccion_t_original']
agronet_anual_corr['delta_rendimiento_t_ha'] = agronet_anual_corr['rendimiento_t_ha'] - agronet_anual_corr['rendimiento_t_ha_original']

display(agronet_anual_corr[mask_2008_cundi])


,anio,departamento,n_municipios,area_cosechada_ha,area_sembrada_ha,produccion_t_original,rendimiento_medio_municipal_reportado,rendimiento_t_ha_original,dif_rendimiento_calculado_vs_reportado,correccion_aplicada,motivo_correccion,produccion_t,rendimiento_t_ha,delta_produccion_t,delta_rendimiento_t_ha
2,2008,Cundinamarca,66,43633.35,48989.09,78254.745626,1.908602,1.793462,-0.115141,1,Outlier de rendimiento de Cundinamarca 2008 corregido con promedio 2007-2009,35732.355721,0.818923,-42522.389905,-0.974539


## Variables derivadas recalculadas despues de la correccion

Una vez corregida la serie, recalculamos las medias historicas por departamento y una medida anual de desviacion respecto a esa media. Esta ultima sigue siendo una variable intermedia util para el pipeline, aunque la definicion final del target se afina en notebooks posteriores.


In [13]:
media_rend = (
    agronet_anual_corr.groupby('departamento', as_index=False)['rendimiento_t_ha']
    .mean()
    .rename(columns={'rendimiento_t_ha': 'rendimiento_medio_t_ha'})
)

media_prod = (
    agronet_anual_corr.groupby('departamento', as_index=False)['produccion_t']
    .mean()
    .rename(columns={'produccion_t': 'produccion_media_t'})
)

agronet_anual_corr = agronet_anual_corr.merge(media_rend, on='departamento', how='left')
agronet_anual_corr = agronet_anual_corr.merge(media_prod, on='departamento', how='left')

agronet_anual_corr['perdida_real_pct'] = (
    (agronet_anual_corr['rendimiento_t_ha'] - agronet_anual_corr['rendimiento_medio_t_ha']) /
    agronet_anual_corr['rendimiento_medio_t_ha'] * 100
)

agronet_anual_corr['evento_perdida_real'] = (agronet_anual_corr['perdida_real_pct'] < -15).astype(int)

display(
    agronet_anual_corr[
        ['anio', 'departamento', 'produccion_t_original', 'produccion_t', 'rendimiento_t_ha_original', 'rendimiento_t_ha', 'rendimiento_medio_t_ha', 'produccion_media_t', 'perdida_real_pct', 'evento_perdida_real', 'correccion_aplicada']
    ].sort_values(['departamento', 'anio']).head(12)
)


,anio,departamento,produccion_t_original,produccion_t,rendimiento_t_ha_original,rendimiento_t_ha,rendimiento_medio_t_ha,produccion_media_t,perdida_real_pct,evento_perdida_real,correccion_aplicada
0,2007,Cundinamarca,33729.143730,33729.143730,0.784083,0.784083,0.925118,30017.744086,-15.245048,1,0
2,2008,Cundinamarca,78254.745626,35732.355721,1.793462,0.818923,0.925118,30017.744086,-11.479062,0,1
4,2009,Cundinamarca,37118.057049,37118.057049,0.853763,0.853763,0.925118,30017.744086,-7.713077,0,0
6,2010,Cundinamarca,37214.800000,37214.800000,0.840743,0.840743,0.925118,30017.744086,-9.120406,0,0
8,2011,Cundinamarca,32780.348700,32780.348700,0.874635,0.874635,0.925118,30017.744086,-5.456866,0,0
10,2012,Cundinamarca,30786.406900,30786.406900,0.828147,0.828147,0.925118,30017.744086,-10.482027,0,0
12,2013,Cundinamarca,24993.744755,24993.744755,0.690641,0.690641,0.925118,30017.744086,-25.345584,1,0
14,2014,Cundinamarca,25118.557250,25118.557250,0.747053,0.747053,0.925118,30017.744086,-19.247836,1,0
16,2015,Cundinamarca,31165.162260,31165.162260,0.913894,0.913894,0.925118,30017.744086,-1.213192,0,0
18,2016,Cundinamarca,31413.339460,31413.339460,0.945781,0.945781,0.925118,30017.744086,2.233614,0,0


## Tabla final a exportar

Definimos explicitamente las columnas que conservara la capa anual limpia para que el siguiente notebook trabaje sobre una estructura estable.


In [14]:
cols_export = [
    'anio',
    'departamento',
    'n_municipios',
    'area_cosechada_ha',
    'area_sembrada_ha',
    'produccion_t_original',
    'rendimiento_t_ha_original',
    'produccion_t',
    'rendimiento_t_ha',
    'correccion_aplicada',
    'motivo_correccion',
    'delta_produccion_t',
    'delta_rendimiento_t_ha',
    'rendimiento_medio_municipal_reportado',
    'dif_rendimiento_calculado_vs_reportado',
    'rendimiento_medio_t_ha',
    'produccion_media_t',
    'perdida_real_pct',
    'evento_perdida_real',
]

agronet_anual_final = agronet_anual_corr[cols_export].sort_values(['departamento', 'anio']).reset_index(drop=True)
print('Shape final de exportacion:', agronet_anual_final.shape)
display(agronet_anual_final.head(12))


Shape final de exportacion: (36, 19)


,anio,departamento,n_municipios,area_cosechada_ha,area_sembrada_ha,produccion_t_original,rendimiento_t_ha_original,produccion_t,rendimiento_t_ha,correccion_aplicada,motivo_correccion,delta_produccion_t,delta_rendimiento_t_ha,rendimiento_medio_municipal_reportado,dif_rendimiento_calculado_vs_reportado,rendimiento_medio_t_ha,produccion_media_t,perdida_real_pct,evento_perdida_real
0,2007,Cundinamarca,65,43017.30000,48195.69,33729.143730,0.784083,33729.143730,0.784083,0,<NA>,0.000000,0.000000,0.629565,0.154518,0.925118,30017.744086,-15.245048,1
1,2008,Cundinamarca,66,43633.35000,48989.09,78254.745626,1.793462,35732.355721,0.818923,1,Outlier de rendimiento de Cundinamarca 2008 corregido con promedio 2007-2009,-42522.389905,-0.974539,1.908602,-0.115141,0.925118,30017.744086,-11.479062,0
2,2009,Cundinamarca,69,43475.84000,48581.30,37118.057049,0.853763,37118.057049,0.853763,0,<NA>,0.000000,0.000000,0.751595,0.102168,0.925118,30017.744086,-7.713077,0
3,2010,Cundinamarca,69,44264.16000,49357.78,37214.800000,0.840743,37214.800000,0.840743,0,<NA>,0.000000,0.000000,0.783393,0.057350,0.925118,30017.744086,-9.120406,0
4,2011,Cundinamarca,69,37478.87000,46123.78,32780.348700,0.874635,32780.348700,0.874635,0,<NA>,0.000000,0.000000,0.854516,0.020120,0.925118,30017.744086,-5.456866,0
5,2012,Cundinamarca,68,37175.06000,45269.86,30786.406900,0.828147,30786.406900,0.828147,0,<NA>,0.000000,0.000000,0.732413,0.095734,0.925118,30017.744086,-10.482027,0
6,2013,Cundinamarca,70,36189.18000,41994.28,24993.744755,0.690641,24993.744755,0.690641,0,<NA>,0.000000,0.000000,0.680867,0.009775,0.925118,30017.744086,-25.345584,1
7,2014,Cundinamarca,69,33623.54000,37996.32,25118.557250,0.747053,25118.557250,0.747053,0,<NA>,0.000000,0.000000,0.736835,0.010217,0.925118,30017.744086,-19.247836,1
8,2015,Cundinamarca,69,34101.49000,37916.38,31165.162260,0.913894,31165.162260,0.913894,0,<NA>,0.000000,0.000000,0.926877,-0.012983,0.925118,30017.744086,-1.213192,0
9,2016,Cundinamarca,69,33214.16263,37679.83,31413.339460,0.945781,31413.339460,0.945781,0,<NA>,0.000000,0.000000,0.958577,-0.012795,0.925118,30017.744086,2.233614,0


## Validaciones finales antes de exportar

Este bloque revisa consistencia basica para no dejar un intermedio roto para el resto del flujo.


In [15]:
assert agronet_anual_final.duplicated(subset=['departamento', 'anio']).sum() == 0, 'Hay duplicados por departamento-anio.'
assert set(agronet_anual_final['departamento'].unique()) == {'Cundinamarca', 'Risaralda'}, 'Departamentos inesperados en la base anual.'
assert agronet_anual_final['anio'].min() == 2007, 'El anio minimo esperado es 2007.'
assert agronet_anual_final['anio'].max() == 2024, 'El anio maximo esperado es 2024.'
assert agronet_anual_final['correccion_aplicada'].sum() == 1, 'Se esperaba exactamente una correccion marcada.'
assert (agronet_anual_final['area_cosechada_ha'] > 0).all(), 'Existen areas cosechadas no positivas.'
assert (agronet_anual_final['produccion_t'] > 0).all(), 'Existen producciones no positivas.'
assert agronet_anual_final['rendimiento_t_ha'].notna().all(), 'Hay rendimientos nulos despues de la construccion.'

print('Validaciones finales superadas correctamente.')


Validaciones finales superadas correctamente.


## Exportacion del archivo intermedio

Se guarda con separador `;` para mantener consistencia con las otras bases del proyecto.


In [16]:
agronet_anual_final.to_csv(OUTPUT_PATH, index=False, sep=';')
print(f'Archivo exportado en: {OUTPUT_PATH}')


Archivo exportado en: C:\Users\crist\Documents\MAESTRIA\PROYECTO_GRADO\BASE_DE_DATOS\INTERMEDIAS\agronet_departamento_anual.csv


## Resumen ejecutivo del notebook

Este bloque deja una lectura corta del resultado para conectar con el siguiente paso del pipeline.


In [17]:
resumen = {
    'archivo_salida': str(OUTPUT_PATH),
    'shape_final': agronet_anual_final.shape,
    'periodo': f"{int(agronet_anual_final['anio'].min())}-{int(agronet_anual_final['anio'].max())}",
    'departamentos': sorted(agronet_anual_final['departamento'].unique().tolist()),
    'n_correcciones': int(agronet_anual_final['correccion_aplicada'].sum()),
    'anios_corregidos': agronet_anual_final.loc[agronet_anual_final['correccion_aplicada'] == 1, ['departamento', 'anio']].to_dict(orient='records'),
    'siguiente_paso': 'usar esta base anual limpia junto con la base clima-satelite para construir la integracion anual y mensual del proyecto',
}

print(json.dumps(resumen, indent=2, ensure_ascii=False))


{
  "archivo_salida": "C:\\Users\\crist\\Documents\\MAESTRIA\\PROYECTO_GRADO\\BASE_DE_DATOS\\INTERMEDIAS\\agronet_departamento_anual.csv",
  "shape_final": [
    36,
    19
  ],
  "periodo": "2007-2024",
  "departamentos": [
    "Cundinamarca",
    "Risaralda"
  ],
  "n_correcciones": 1,
  "anios_corregidos": [
    {
      "departamento": "Cundinamarca",
      "anio": 2008
    }
  ],
  "siguiente_paso": "usar esta base anual limpia junto con la base clima-satelite para construir la integracion anual y mensual del proyecto"
}


## Siguiente notebook recomendado

El siguiente paso natural del flujo es construir `03_validacion_clima_satelite.ipynb` para revisar la base mensual de entrada antes de integrarla con esta nueva capa anual.
